# Silver Products

Clean and transform products data from bronze to silver layer with data quality checks: clean data, validate keys, and remove duplicates.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/SilverLoader"

In [0]:
silver_table_name = get_table_name("products", "silver")
bronze_table_name = get_table_name("products", "bronze")

In [0]:
spark.read.table(bronze_table_name).display()

In [0]:
from pyspark.sql import functions as F

# --- Transform function for silver products ---
def transform_products(bronze_df):
    silver_df = (bronze_df
        .select(
            F.col("product_id"),
            clean_text("product_name").alias("product_name"),
            clean_text("category").alias("category"),
            clean_text("sub_category").alias("sub_category"),
            clean_text("brand").alias("brand"),
            F.col("price"),
            F.col("cost"),
            F.col("created_at"),
            F.col("updated_at"),
            F.col("_bronze_ingestion_ts")
        )
    )
    silver_df = validate_keys(silver_df, ["product_id"])
    silver_df = remove_duplicates(silver_df, ["product_id"], order_col="updated_at")
    silver_df = drop_bronze_metadata(silver_df)
    return silver_df

result = incremental_silver_load(
    bronze_table_name=bronze_table_name,
    silver_table_name=silver_table_name,
    key_columns=["product_id"],
    transform_fn=transform_products,
    update_condition="source.updated_at >= target.updated_at"
)

print(result)

In [0]:
display(spark.read.table(silver_table_name))